In [2]:
# NOTEBOOK 03: Build Data Warehouse (Fact + Dimension Tables)
# Purpose: Create dimensional schema for analytics
# ==============================================================

print("🏗️ Building Data Warehouse Schema...\n")

# ============================================================
# STEP 1: Create Dimension Tables
# ============================================================

# DIM_COMPANIES
print("Creating dim_companies...")
spark.sql("""
    CREATE OR REPLACE TABLE dim_companies AS
    SELECT 
        ROW_NUMBER() OVER (ORDER BY company_name) AS company_id,
        company_name,
        'Unknown' AS industry,
        CURRENT_TIMESTAMP() AS created_date
    FROM stg_companies_cleaned
""")
print("✅ dim_companies created\n")

# DIM_JOB_TITLES
print("Creating dim_job_titles...")
spark.sql("""
    CREATE OR REPLACE TABLE dim_job_titles AS
    SELECT 
        ROW_NUMBER() OVER (ORDER BY job_title) AS job_title_id,
        job_title,
        CASE 
            WHEN job_title LIKE '%EDI%' OR job_title LIKE '%Integration%' OR job_title LIKE '%Clearinghouse%' THEN 'EDI & Integration'
            WHEN job_title LIKE '%BI%' OR job_title LIKE '%Analytics%' OR job_title LIKE '%Business Intelligence%' THEN 'Business Intelligence'
            WHEN job_title LIKE '%Data Engineer%' OR job_title LIKE '%ETL%' OR job_title LIKE '%Data Pipeline%' THEN 'Data Engineering'
            WHEN job_title LIKE '%Data Scientist%' OR job_title LIKE '%Machine Learning%' THEN 'Data Science & ML'
            WHEN job_title LIKE '%Analyst%' THEN 'Analysis'
            WHEN job_title LIKE '%Architect%' THEN 'Architecture'
            WHEN job_title LIKE '%Manager%' OR job_title LIKE '%Lead%' THEN 'Leadership'
            ELSE 'Other'
        END AS job_category,
        CURRENT_TIMESTAMP() AS created_date
    FROM (
        SELECT DISTINCT job_title
        FROM stg_jobs_cleaned
    )
""")
print("✅ dim_job_titles created\n")

# DIM_SKILLS
print("Creating dim_skills...")
spark.sql("""
    CREATE OR REPLACE TABLE dim_skills AS
    SELECT 
        ROW_NUMBER() OVER (ORDER BY skill) AS skill_id,
        skill AS skill_name,
        CASE 
            WHEN skill IN ('SQL', 'Python', 'Spark', 'dbt') THEN 'Data Transformation'
            WHEN skill IN ('Power BI', 'Tableau') THEN 'Business Intelligence'
            WHEN skill IN ('Azure', 'AWS', 'Redshift', 'BigQuery') THEN 'Cloud & Data Warehousing'
            WHEN skill IN ('X12 EDI', 'EDI', 'HL7') THEN 'Healthcare EDI'
            WHEN skill IN ('Machine Learning', 'TensorFlow', 'PyTorch') THEN 'Machine Learning'
            ELSE 'Technical'
        END AS skill_category,
        CURRENT_TIMESTAMP() AS created_date
    FROM stg_skills_extracted
""")
print("✅ dim_skills created\n")

# DIM_LOCATIONS
print("Creating dim_locations...")
spark.sql("""
    CREATE OR REPLACE TABLE dim_locations AS
    WITH location_split AS (
        SELECT DISTINCT
            location,
            SPLIT(location, ' ')[0] AS city,
            SPLIT(location, ' ')[SIZE(SPLIT(location, ' ')) - 1] AS state
        FROM stg_jobs_cleaned
        WHERE location IS NOT NULL
    )
    SELECT 
        ROW_NUMBER() OVER (ORDER BY city, state) AS location_id,
        city,
        state,
        location AS full_location,
        'USA' AS country,
        CURRENT_TIMESTAMP() AS created_date
    FROM location_split
""")
print("✅ dim_locations created\n")

# ============================================================
# STEP 2: Create Fact Table
# ============================================================

print("Creating fact_job_postings...")
spark.sql("""
    CREATE OR REPLACE TABLE fact_job_postings AS
    SELECT 
        j.job_id,
        c.company_id,
        jt.job_title_id,
        l.location_id,
        j.salary_min,
        j.salary_max,
        ROUND((j.salary_min + j.salary_max) / 2.0, 2) AS avg_salary,
        j.posted_date,
        j.ingestion_date,
        CURRENT_DATE() AS fact_date
    FROM stg_jobs_cleaned j
    LEFT JOIN dim_companies c ON TRIM(j.company_name) = TRIM(c.company_name)
    LEFT JOIN dim_job_titles jt ON TRIM(j.job_title) = TRIM(jt.job_title)
    LEFT JOIN dim_locations l ON TRIM(j.location) = TRIM(l.full_location)
""")
print("✅ fact_job_postings created\n")

# ============================================================
# STEP 3: Create Bridge Table (Jobs to Skills - Many-to-Many)
# ============================================================

print("Creating bridge_job_skills...")
spark.sql("""
    CREATE OR REPLACE TABLE bridge_job_skills AS
    SELECT DISTINCT
        j.job_id,
        ds.skill_id,
        ds.skill_name,
        CURRENT_TIMESTAMP() AS created_date
    FROM stg_jobs_cleaned j
    CROSS JOIN dim_skills ds
    WHERE LOWER(j.job_description) LIKE CONCAT('%', LOWER(ds.skill_name), '%')
""")
print("✅ bridge_job_skills created\n")

# ============================================================
# STEP 4: Display Schema Summary
# ============================================================

print("\n" + "="*60)
print("📊 DATA WAREHOUSE SUMMARY")
print("="*60 + "\n")

spark.sql("SELECT 'dim_companies' as table_name, COUNT(*) as record_count FROM dim_companies").show()
spark.sql("SELECT 'dim_job_titles' as table_name, COUNT(*) as record_count FROM dim_job_titles").show()
spark.sql("SELECT 'dim_skills' as table_name, COUNT(*) as record_count FROM dim_skills").show()
spark.sql("SELECT 'dim_locations' as table_name, COUNT(*) as record_count FROM dim_locations").show()
spark.sql("SELECT 'fact_job_postings' as table_name, COUNT(*) as record_count FROM fact_job_postings").show()
spark.sql("SELECT 'bridge_job_skills' as table_name, COUNT(*) as record_count FROM bridge_job_skills").show()

# ============================================================
# STEP 5: Sample Data Preview
# ============================================================

print("\n" + "="*60)
print("📋 SAMPLE DATA")
print("="*60 + "\n")

print("--- Fact Table Sample ---")
spark.sql("SELECT * FROM fact_job_postings LIMIT 3").show()

print("\n--- Skill Dimension Sample ---")
spark.sql("SELECT * FROM dim_skills LIMIT 5").show()

print("\n--- Job Skills Bridge Sample ---")
spark.sql("SELECT * FROM bridge_job_skills LIMIT 5").show()

print("\n✅ Phase 3 Complete: Data Warehouse Ready!")
print("\n🎯 Next Step: Connect to Power BI and build dashboards")

StatementMeta(, ea581b98-0cc1-4214-ab9f-a4350f840463, 4, Finished, Available, Finished, False)

🏗️ Building Data Warehouse Schema...

Creating dim_companies...
✅ dim_companies created

Creating dim_job_titles...
✅ dim_job_titles created

Creating dim_skills...
✅ dim_skills created

Creating dim_locations...
✅ dim_locations created

Creating fact_job_postings...
✅ fact_job_postings created

Creating bridge_job_skills...
✅ bridge_job_skills created


📊 DATA WAREHOUSE SUMMARY

+-------------+------------+
|   table_name|record_count|
+-------------+------------+
|dim_companies|          47|
+-------------+------------+

+--------------+------------+
|    table_name|record_count|
+--------------+------------+
|dim_job_titles|          47|
+--------------+------------+

+----------+------------+
|table_name|record_count|
+----------+------------+
|dim_skills|           5|
+----------+------------+

+-------------+------------+
|   table_name|record_count|
+-------------+------------+
|dim_locations|          23|
+-------------+------------+

+-----------------+------------+
|       ta